# Executive B2B Energy Churn Prediction & Financial Risk Optimization

**Author:** Giovanni (Statistics Student, UCV)  
**Role:** Quantitative Data Science Consultant  
**Target Stakeholders:** Estelle (Senior Data Scientist, BCG) & Associate Director (AD)  

---

## Executive Overview
This notebook presents an end-to-end quantitative Machine Learning framework for predicting B2B customer churn at PowerCo using `RandomForestClassifier`. The analytical pipeline addresses severe class imbalance (~90% retention, 10% churn) and incorporates asymmetric financial loss functions to evaluate commercial discount strategies.

---

## Phase 1: Environment Setup & Data Ingestion

We import core computational libraries (`pandas`, `numpy`, `sklearn`) and load the engineered dataset `data_for_predictions.csv` provided by Estelle.

In [ ]:
# Import required computational and plotting libraries
import pandas as pd
pd.set_option('display.max_columns', None)
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns
import sklearn

### 1.1 Dataset Loading & Cleaning
We load `data_for_predictions.csv` and drop non-predictive index columns.

In [ ]:
# Load prediction dataset and drop index column
df = pd.read_csv("data_for_predictions.csv")
df = df.drop(columns=['Unnamed: 0'])
df.head()

### 1.2 Structure & Imbalance Verification
We inspect data types and verify the baseline class distribution in `churn`.

In [ ]:
# Display DataFrame summary
df.info()

In [ ]:
# Verify target class proportion (severe 90/10 imbalance)
df['churn'].value_counts(normalize=True)

---
## Phase 2: Model Architecture & Stratified Training

### 2.1 Feature Matrix (X) and Target Vector (y) Separation
We isolate target variable `churn` as `y` and drop arbitrary string identifiers (`id`) alongside `churn` to form feature matrix `X`.

### 2.2 Stratified Train-Test Split
We apply `train_test_split` with `stratify=y` to preserve the 90.3% / 9.7% class proportion across both training (80%) and testing (20%) sets.

### 2.3 RandomForestClassifier Model Instantiation
We fit `RandomForestClassifier` with `class_weight='balanced'` to penalize minority class errors inversely proportional to class frequencies.

In [ ]:
# Import model selection and ensemble modules
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# Separate feature matrix X and target vector y
X = df.drop(columns=['id','churn'])
y = df['churn']

# Perform stratified 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,y, test_size=0.2, random_state=42, stratify=y
)

# Instantiate and fit Random Forest with balanced class weights
RF = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42
)
RF.fit(X_train, y_train)

---
## Phase 3: Model Evaluation in Extremistan

### Executive Question 1 Response (Estelle / BCG Inquiry):
> **"Why did you choose the evaluation metrics that you used? Please elaborate on your choices."**

#### Technical Justification:
1. **Rejection of Accuracy (90.3% Fallacy):** In heavily imbalanced datasets (~90% retention, 10% churn), a naive baseline model predicting zero churn achieves 90.3% Accuracy while failing to detect a single churner. Accuracy provides a false sense of security.
2. **Confusion Matrix:** Decomposes classification performance into True Negatives (TN), False Positives (FP), False Negatives (FN), and True Positives (TP), enabling exact financial error mapping.
3. **Recall (Sensitivity):** Measures the proportion of actual churners captured by the model. This is the primary operational metric because missing a churner results in a 100% loss of net margin.
4. **Precision:** Measures the accuracy of positive churn alarms to prevent wasting discount budget on non-churners.
5. **ROC-AUC Score (0.690):** Evaluates overall ranking and class separability across all possible decision thresholds.

In [ ]:
# Import metrics module and generate predictions
from sklearn.metrics import confusion_matrix, roc_auc_score, classification_report

y_pred = RF.predict(X_test)
y_pred_proba = RF.predict_proba(X_test)[:,1]

In [ ]:
# Print Confusion Matrix
print(f'Confusion Matrix:\n {confusion_matrix(y_test, y_pred)}')

In [ ]:
# Print Classification Report
print(f'Clasification Report:\n {classification_report(y_test, y_pred)}')

In [ ]:
# Print ROC-AUC Score
print(f'ROC-AUC Score:\n {roc_auc_score(y_test, y_pred_proba)}')

---
## Phase 4: Asymmetric Financial Risk & Threshold Optimization

### Executive Question 2 Response (Estelle / BCG Inquiry):
> **"Do you think that the model performance is satisfactory? Give justification for your answer."**

#### Executive Justification:
- **Under Default Threshold (0.50): NOT SATISFACTORY.**  
  At the standard 0.50 threshold, the model exhibits operational blindness, capturing only 18% of actual churners (Recall = 18%) and leaving 234 undetected churners (losing -$47,591.80 in net margin).
- **Under Optimized Financial Threshold (0.25): HIGHLY SATISFACTORY.**  
  Because losing a customer (100% margin loss) is 5 times more expensive than offering an unnecessary discount (20% margin discount), requiring 50% voting confidence is commercially sub-optimal. Sweeping decision thresholds reveals that setting the decision threshold to **p = 0.25** cuts unseen churn losses in half, protecting **+$27,420.58 in net margin** compared to the zero-intervention baseline.

In [ ]:
# Execute financial threshold sweep loop across thresholds p in [0.10, 0.80]
results_sweep = []
for th in np.arange(0.10, 0.85, 0.05):
    y_pred_th = (y_pred_proba >= th).astype(int)
    fin = pd.DataFrame({'net_margin': X_test['net_margin'], 'actual': y_test, 'predicted': y_pred_th})
    
    tp = fin[(fin['actual'] == 1) & (fin['predicted'] == 1)]['net_margin'].sum()
    fp = fin[(fin['actual'] == 0) & (fin['predicted'] == 1)]['net_margin'].sum()
    fn = fin[(fin['actual'] == 1) & (fin['predicted'] == 0)]['net_margin'].sum()
    
    saved = 0.80 * tp
    giveaway = 0.20 * fp
    losses = 1.00 * fn
    net = saved - giveaway - losses
    
    results_sweep.append({
        'Threshold': round(th, 2),
        'Saved_TP': round(saved, 2),
        'Giveaway_FP': round(giveaway, 2),
        'Losses_FN': round(losses, 2),
        'Net_Impact': round(net, 2)
    })

sweep_df = pd.DataFrame(results_sweep)
sweep_df.sort_values(by='Net_Impact', ascending=False)

---
## Phase 5: Feature Importances & Econometric Insights

We extract impurity-based feature importances (`RF.feature_importances_`) to identify top drivers of customer churn.

In [ ]:
# Extract top 10 feature importances
importances = pd.Series(RF.feature_importances_, index=X.columns)
top_10 = importances.sort_values(ascending=False).head(10)
print("Top 10 Feature Importances:")
print(top_10)

---
## Final Executive Summary & Commercial Recommendations

1. **Primary Churn Drivers:** Churn is dominated by volumetric consumption (`cons_12m`) and account margin metrics (`margin_net_pow_ele`, `margin_gross_pow_ele`, `net_margin`), proving that account size and overall margin exposure dictate customer churn rather than raw unweighted price changes.
2. **Optimal Commercial Strategy:** Deploy the Random Forest model using the **p = 0.25 probability threshold** to issue 20% discount offers, protecting **+$27,420.58 in net margin** for PowerCo.